In [1]:
# importamos librerías
# para manipulación de datos
import pandas as pd
import numpy as np
# para trabajar con APIs
import requests
# para gestionar archivos y directorios
import os
import zipfile
# para manipular pdfs
import camelot
# para trabajar con fechas
from datetime import datetime

In [ ]:
def descargar_datos_cmadrid(url, carpeta_salida, formato = None, mes = False, dia = False):
    """
    Descarga archivos desde una URL y los guarda en la carpeta especificada.

    Args:
        url (str): URL de la API que proporciona los datos a descargar.
        carpeta_salida (str): Ruta de la carpeta donde se guardarán los archivos.
        formato (str, opcional): Formato de archivo deseado (ej. "csv"). Si es None, descarga cualquier formato disponible.
        mes (bool, opcional): Si es True, incluye el mes en el nombre del archivo.
        dia (bool, opcional): Si es True, incluye el día en el nombre del archivo.

    Returns:
        list: Lista con los nombres de los archivos descargados exitosamente.

    Raises:
        requests.exceptions.RequestException: Si hay un problema con la solicitud HTTP.
        Exception: Para otros errores inesperados en el proceso de descarga.

    """

    hoy =  datetime.now().date()
    response = requests.get(url)
    archivos_descargados = []
    if response.status_code == 200:
        data = response.json()
        info = data.get("result", {}).get("resources")
        archivos_guardados = 0
        archivos_totales = len(info)
        formato_usuario = formato.lower() if formato else None
        for elemento in info:
            formato_archivo = elemento.get("format").lower()
            nombre = elemento.get("name")
            if formato_usuario is None or formato_archivo == formato_usuario:
                datos = elemento.get("url")
                response = requests.get(datos)
                # Verificar si la descarga fue exitosa
                if response.status_code == 200:
                    if not mes and not dia:
                        nombre = f'{carpeta_salida}/cmadrid_{nombre}.{formato_archivo}'
                    elif mes and not dia:
                        nombre = f'{carpeta_salida}/cmadrid_{hoy.month}_{hoy.year}.{formato_archivo}'
                    elif dia and not mes:
                        nombre = f'{carpeta_salida}/cmadrid_{hoy}.{formato_archivo}'
                    # Guardar el archivo en tu computadora
                    with open(nombre, 'wb') as file:
                        file.write(response.content) 
                    archivos_guardados +=1 
                    archivos_descargados.append(elemento["name"])   
            else:
                    print(f"No se descargó el archivo {elemento["name"]}.{elemento["format"]}. Código de estado: {response.status_code}")
    print(f"{archivos_guardados} de {archivos_totales} archivos guardados exitosamente.")
    return archivos_descargados

In [67]:
historicos_cmadrid = descargar_datos_cmadrid("https://datos.comunidad.madrid/api/3/action/package_show?id=calidad_aire_datos_historico")
historicos_madrid = historicos_cmadrid[1:]

22 de 22 archivos guardados exitosamente.


In [ ]:
def descargar_datos_madrid(url, carpeta_salida):
    """
    Descarga los datos de calidad del aire de Madrid y los guarda en archivos locales.

    Args:
        url (str): URL del servicio que proporciona los datos de calidad del aire.
        carpeta_salida: ruta de la carpeta donde se guardarán los archivos.

    Returns:
        None: Guarda los archivos en la carpeta correspondiente.
    """
    # Obtener la fecha actual
    hoy = datetime.now().date()
    archivos_dh = []

    # Realizar la solicitud HTTP y verificar la respuesta
    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()
        elementos = data.get("result", {}).get("items")

        # Recorrer los elementos disponibles en la respuesta
        for elemento in elementos:
            titulo = elemento["title"]
            datos = elemento.get("distribution")

            # Procesar los datos según el título del elemento
            if titulo == "Calidad del aire. Estaciones de control":
                for i in datos:
                    formato = i.get("format", {}).get("value").split('/')[-1].lower()
                    if formato == "csv":
                        archivo = i.get("accessURL")
                        print(f"Descargando: {archivo} como madrid_{titulo}.{formato}")
                        response_est = requests.get(archivo)
                        if response_est.status_code == 200:
                            with open(f'{carpeta_salida}/madrid_{titulo}.{formato}', 'wb') as file:
                                file.write(response_est.content)
                            print("Archivo guardado exitosamente.")
                        else:
                            print(f"No se pudo descargar el archivo {titulo}. Código de estado: {response.status_code}")

            elif titulo == "Calidad del aire. Datos en tiempo real acumulado":
                for i in datos:
                    formato = i.get("format", {}).get("value").split('/')[-1]
                    if "csv" in i.get("title"):
                        archivo = i.get("accessURL")
                        print(f"Descargando: {archivo} como madrid_{hoy}.{formato}")
                        response_tra = requests.get(archivo)
                        if response_tra.status_code == 200:
                            with open(f'data/raw/madrid_{hoy}.{formato}', 'wb') as file:
                                file.write(response_tra.content)
                            print("Archivo guardado exitosamente.")
                        else:
                            print(f"No se pudo descargar el archivo {titulo}. Código de estado: {response.status_code}")

            elif titulo == "Calidad del aire. Datos horarios desde 2001":
                for i in datos:
                    formato = i.get("format", {}).get("value").split('/')[-1]
                    archivo = i.get("accessURL")
                    print(f"Descargando: {archivo} como madrid_{i['title']}.{formato}")
                    response_dh = requests.get(archivo)
                    if response_dh.status_code == 200:
                        with open(f'data/raw/madrid_{i['title']}.{formato}', 'wb') as file:
                            file.write(response_dh.content)
                        archivos_dh.append(i["title"])
                        print("Archivo guardado exitosamente.")
                    else:
                        print(f"No se pudo descargar el archivo {i['title']}. Código de estado: {response.status_code}")
    return archivos_dh


In [18]:
archivos_madrid= descargar_datos_madrid("https://datos.madrid.es/egob/catalogo/keyword/aire.json")

Descargando: https://datos.madrid.es/egob/catalogo/300755-12751583-calidad-aire-tiempo-real-acumula.csv como madrid_2025-05-02.csv
Archivo guardado exitosamente.
Descargando: https://datos.madrid.es/egob/catalogo/201200-10306321-calidad-aire-horario.zip como madrid_2025.zip
Archivo guardado exitosamente.
Descargando: https://datos.madrid.es/egob/catalogo/201200-10306320-calidad-aire-horario.zip como madrid_2024.zip
Archivo guardado exitosamente.
Descargando: https://datos.madrid.es/egob/catalogo/201200-10306319-calidad-aire-horario.zip como madrid_2023.zip
Archivo guardado exitosamente.
Descargando: https://datos.madrid.es/egob/catalogo/201200-10306318-calidad-aire-horario.zip como madrid_2022.zip
Archivo guardado exitosamente.
Descargando: https://datos.madrid.es/egob/catalogo/201200-10306317-calidad-aire-horario.zip como madrid_2021.zip
Archivo guardado exitosamente.
Descargando: https://datos.madrid.es/egob/catalogo/201200-10306316-calidad-aire-horario.zip como madrid_2020.zip
Archi

In [ ]:
def procesar_archivos_zip(archivos_dh, carpeta_salida):
    """
    Procesa archivos ZIP que contienen datos en formato CSV, extrae los archivos,
    los concatena y guarda el resultado en un nuevo CSV.

    Args:
        archivos_dh (list): Lista con los nombres de los archivos ZIP a procesar.
        carpeta_salida (str): Ruta de la carpeta donde se guardarán los archivos procesados.

    Returns:
        None: Guarda los archivos CSV concatenados en la carpeta indicada.
    """
    # Crear la carpeta de salida si no existe
    os.makedirs(carpeta_salida, exist_ok=True)

    # Diccionarios para almacenar listas de DataFrames y nombres de CSV
    listas_dfs = {}
    listas_csv = {}

    for archivo in archivos_dh:
        ruta_zip = os.path.join(carpeta_salida, f"madrid_{archivo}.zip")  # Construir la ruta al archivo ZIP
        listas_dfs[archivo] = []  # Inicializar una lista de DataFrames para este archivo
        listas_csv[archivo] = []  # Inicializar una lista de nombres de CSV para este archivo

        # Abrir el ZIP y cargar los CSV
        with zipfile.ZipFile(ruta_zip, 'r') as zip_ref:
            listas_csv[archivo] = [nombre for nombre in zip_ref.namelist() if nombre.endswith(".csv")]

            for nombre_csv in listas_csv[archivo]:
                with zip_ref.open(nombre_csv) as f:
                    df = pd.read_csv(f, sep=";", index_col=0)
                    listas_dfs[archivo].append(df)

        # Concatenar los DataFrames de la lista en uno solo
        df_concatenado = pd.concat(listas_dfs[archivo], ignore_index=True)

        # Guardar el DataFrame concatenado en un archivo CSV
        ruta_salida = os.path.join(carpeta_salida, f"madrid_{archivo}.csv")
        df_concatenado.to_csv(ruta_salida, sep=";", index=False)

        print(f"Archivo concatenado guardado en: {ruta_salida}")

        # Eliminar el archivo ZIP
        os.remove(ruta_zip)
        print(f"Archivo ZIP eliminado: {ruta_zip}")

In [21]:
madrid_procesado=procesar_archivos_zip(archivos_dh, carpeta_salida="../datos")

Archivo concatenado guardado en: ../datos\madrid_2025.csv
Archivo ZIP eliminado: ../datos\madrid_2025.zip
Archivo concatenado guardado en: ../datos\madrid_2024.csv
Archivo ZIP eliminado: ../datos\madrid_2024.zip
Archivo concatenado guardado en: ../datos\madrid_2023.csv
Archivo ZIP eliminado: ../datos\madrid_2023.zip
Archivo concatenado guardado en: ../datos\madrid_2022.csv
Archivo ZIP eliminado: ../datos\madrid_2022.zip
Archivo concatenado guardado en: ../datos\madrid_2021.csv
Archivo ZIP eliminado: ../datos\madrid_2021.zip
Archivo concatenado guardado en: ../datos\madrid_2020.csv
Archivo ZIP eliminado: ../datos\madrid_2020.zip
Archivo concatenado guardado en: ../datos\madrid_2019.csv
Archivo ZIP eliminado: ../datos\madrid_2019.zip
Archivo concatenado guardado en: ../datos\madrid_2018.csv
Archivo ZIP eliminado: ../datos\madrid_2018.zip
Archivo concatenado guardado en: ../datos\madrid_2017.csv
Archivo ZIP eliminado: ../datos\madrid_2017.zip
Archivo concatenado guardado en: ../datos\madr

In [ ]:
def extraer_tabla_pdf(archivo_pdf, pagina, carpeta_salida):
    """
    Extrae la primera tabla de una página específica de un archivo PDF y la guarda como CSV.

    Args:
        archivo_pdf (str): Ruta del archivo PDF del cual se extraerán los datos.
        pagina (str): Número de la página donde se encuentra la tabla a extraer.
        carpeta_salida (str): Ruta donde se guardará el archivo CSV.

    Returns:
        bool: True si se extrajo y guardó la tabla correctamente, False si no se encontraron tablas.

    Raises:
        FileNotFoundError: Si el archivo PDF no existe.
        Exception: Para errores inesperados en la extracción de la tabla.
    """
    try:
        # Extraer tablas de la página indicada
        tablas = camelot.read_pdf(archivo_pdf, pages=pagina)

        # Verificar si se extrajeron tablas
        if len(tablas) > 0:
            tablas[0].to_csv(f"{carpeta_salida}/tabla_contaminantes.csv")  # Guardar la primera tabla como CSV
            print(f"✅ Tabla guardada en {carpeta_salida}")
            return True
        else:
            print("⚠️ No se encontraron tablas en la página indicada.")
            return False

    except FileNotFoundError:
        print(f"❌ Error: No se encontró el archivo {archivo_pdf}")
    except Exception as e:
        print(f"❌ Error inesperado: {e}")